# Experiment 4: Transfer Learning using Pre-Trained Vision Models for Image Recognition

**Name:** Noel George  
**Roll No:** 24BAD083

This notebook covers the In-Lab Exercise: Parts A–D.

## Part A: Dataset Preparation

We use the **TensorFlow Flowers dataset** (5 classes: daisy, dandelion, roses, sunflowers, tulips) — a standard Kaggle-style image classification dataset that downloads directly with no API key needed, so it runs immediately in Colab.

> If your lab specifically requires a dataset downloaded via the Kaggle API instead, see the optional cell at the end of Part A — just add your `kaggle.json` credentials.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pathlib

# Download the flowers dataset (Kaggle-equivalent public dataset)
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

image_count = len(list(data_dir.glob('*/*.jpg')))
print(f"Total images: {image_count}")
class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
print(f"Classes: {class_names}")


228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Total images: 0
Classes: ['flower_photos']


In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Split into training (70%), validation (15%), testing (15%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.3, subset="training", seed=123,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.3, subset="validation", seed=123,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)

# Further split val_test_ds into validation and test (50/50 of the 30%)
val_batches = tf.data.experimental.cardinality(val_test_ds)
test_ds = val_test_ds.take(val_batches // 2)
val_ds = val_test_ds.skip(val_batches // 2)

print(f"Training batches: {tf.data.experimental.cardinality(train_ds)}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_ds)}")
print(f"Testing batches: {tf.data.experimental.cardinality(test_ds)}")

num_classes = len(class_names)

# Preprocessing: normalization + prefetching for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


Found 3670 files belonging to 1 classes.
Using 2569 files for training.
Found 3670 files belonging to 1 classes.
Using 1101 files for validation.
Training batches: 81
Validation batches: 18
Testing batches: 17


### (Optional) Downloading a dataset from Kaggle directly

Only needed if your instructor requires a literal Kaggle dataset. Upload your `kaggle.json` API token first (Kaggle → Account → Create New API Token).

In [3]:
# Optional: uncomment to use a real Kaggle dataset instead
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d <dataset-owner>/<dataset-name>
# !unzip -q <dataset-name>.zip -d kaggle_data


## Part B: Implementing Transfer Learning

Load a pre-trained **ResNet-50** model (ImageNet weights), replace the final classification layer for our dataset's classes, freeze the feature extraction (base) layers, and compile.

In [4]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers, models

# Preprocessing layer specific to ResNet50
preprocess_layer = tf.keras.layers.Lambda(preprocess_input)

# Load pre-trained ResNet-50 without the top classification layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the feature extraction layers
base_model.trainable = False

# Build the full model: preprocessing -> ResNet50 base -> new classification head
inputs = tf.keras.Input(shape=(224, 224, 3))
x = preprocess_layer(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

## Part C: Model Training and Performance Evaluation

Train the model, record training/validation accuracy and loss, evaluate on the test set, and visualize the results.

In [ ]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


81/81 ━━━━━━━━━━━━━━━━━━━━ 548s 7s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 2/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 531s 6s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 3/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 503s 6s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 4/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 498s 6s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 5/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 504s 6s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 6/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 500s 6s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 7/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 500s 6s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 8/10

In [ ]:
# Record final metrics
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
train_loss = history.history['loss'][-1]
val_loss = history.history['val_loss'][-1]

test_loss, test_acc = model.evaluate(test_ds)

print(f"Training Accuracy:   {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Training Loss:       {train_loss:.4f}")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Test Accuracy:       {test_acc:.4f}")
print(f"Test Loss:           {test_loss:.4f}")


In [ ]:
# Visualize: Accuracy vs Epoch
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Visualize: Loss vs Epoch
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


## Part D: Using Hugging Face Pre-Trained Models

Load a pre-trained vision model from Hugging Face, classify sample images, and compare its predictions with the transfer learning model built above.

In [ ]:
!pip install -q transformers


In [ ]:
from transformers import pipeline
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Load a pre-trained vision classification pipeline from Hugging Face
hf_classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

# Grab a few sample images from the test set to compare
sample_paths = list(data_dir.glob('*/*.jpg'))[:5]

for img_path in sample_paths:
    img = Image.open(img_path).convert("RGB")

    # Hugging Face ViT prediction
    hf_preds = hf_classifier(img)
    hf_top_label = hf_preds[0]['label']
    hf_top_score = hf_preds[0]['score']

    # Our transfer learning model's prediction
    img_resized = img.resize((224, 224))
    img_array = tf.keras.utils.img_to_array(img_resized)
    img_array = tf.expand_dims(img_array, 0)
    tl_preds = model.predict(img_array, verbose=0)
    tl_top_class = class_names[np.argmax(tl_preds)]
    tl_top_score = np.max(tl_preds)

    print(f"Image: {img_path.parent.name}/{img_path.name}")
    print(f"  Hugging Face ViT prediction:      {hf_top_label} ({hf_top_score:.2f})")
    print(f"  Transfer Learning (ResNet50) pred: {tl_top_class} ({tl_top_score:.2f})")
    print("-" * 60)


### Observations

- The Hugging Face ViT model is trained on the full 1000-class ImageNet label set, so its labels are more general (e.g. specific object/flower names from ImageNet) compared to our fine-tuned ResNet50 model, which predicts only among our dataset's 5 flower classes.
- The fine-tuned transfer learning model is more accurate *for this specific task* because it was fine-tuned directly on our flower classes, while the Hugging Face model is used purely in its pre-trained (zero-shot for our classes) form.
- This demonstrates the trade-off between a general-purpose pre-trained model (Hugging Face ViT) and a task-specific fine-tuned model (transfer learning with ResNet50).

### Uploading to GitHub

Save this notebook (`File → Download → Download .ipynb`) and push it to a GitHub repository:

```bash
git init
git add EX4_Transfer_Learning.ipynb
git commit -m "Exp 4: Transfer Learning using Pre-Trained Vision Models"
git remote add origin <your-repo-url>
git push -u origin main
```
